In [3]:
# -*- coding: utf-8 -*-                                   # Define a codificação do arquivo como UTF-8
# Script: Extração de voltas (laps) FastF1 2022–2024      # Descrição breve do que o script faz                             

from pathlib import Path                                   # Importa Path para manipular caminhos de forma robusta
import pandas as pd                                        # Importa pandas para análise e manipulação tabular
import numpy as np                                         # Importa numpy para utilidades numéricas/NaN
import fastf1                                             # Importa FastF1 (API de dados públicos da F1)

# =========================
# PARÂMETROS GERAIS
# =========================

YEARS = [2022, 2023, 2024]                                 # Define os anos a processar (fixo para 2022–2024 conforme pedido)

PROJ = Path.cwd().resolve()                                 # Define a pasta base como o diretório atual do processo
DATA_DIR = PROJ / "data"                                    # Define a pasta "data" para saídas
DATA_DIR.mkdir(exist_ok=True)                               # Garante que "data" exista (cria se não existir)

INTERIM = DATA_DIR / "interim"                              # Subpasta para arquivos intermediários
INTERIM.mkdir(parents=True, exist_ok=True)                  # Garante que "data/interim" exista

CACHE_DIR = PROJ / "fastf1_cache"                           # Pasta de cache do FastF1 (evita re-download)
CACHE_DIR.mkdir(exist_ok=True)                              # Garante que a pasta de cache exista
fastf1.Cache.enable_cache(str(CACHE_DIR))                   # Habilita o cache do FastF1 naquele caminho

# =========================
# CIRCUITOS ALVO
# (mapeamento robusto por Location e por substring do EventName)
# =========================

TARGETS = {                                                 # Dicionário mapeando o "rótulo" do circuito aos critérios de busca
    "Bahrein":   {"location": ["sakhir", "bahrain"],            "event_sub": ["bahrain"]},          # Sakhir/Bahrain GP
    "Jeddah":    {"location": ["jeddah"],                       "event_sub": ["saudi"]},            # Saudi Arabian GP
    "Australia": {"location": ["melbourne", "australia"],       "event_sub": ["australian"]},       # Australian GP
    "Baku":      {"location": ["baku"],                         "event_sub": ["azerbaijan"]},       # Azerbaijan GP
    "Miami":     {"location": ["miami"],                        "event_sub": ["miami"]},            # Miami GP
    "Monza":     {"location": ["monza"],                        "event_sub": ["italian"]},          # Italian GP (Monza)
    "Singapura": {"location": ["singapore"],                    "event_sub": ["singapore"]},        # Singapore GP
    "Suzuka":    {"location": ["suzuka"],                       "event_sub": ["japan"]},            # Japanese GP
    "COTA":      {"location": ["austin"],                       "event_sub": ["united states"]},    # United States GP (COTA/Austin)
    "Mexico":    {"location": ["mexico city"],                  "event_sub": ["mexico"]},           # Mexico City GP
    "Brasil":    {"location": ["são paulo", "sao paulo"],       "event_sub": ["sao paulo", "brazil"]},  # São Paulo GP (Interlagos)
    "Abu Dhabi": {"location": ["abu dhabi", "yas marina"],      "event_sub": ["abu dhabi"]},        # Abu Dhabi GP
}

# =========================
# CARREGA CALENDÁRIOS POR ANO
# =========================

all_events = []                                             # Lista para acumular os calendários de cada ano

for y in YEARS:                                             # Itera pelos anos desejados
    cal = fastf1.get_event_schedule(y, include_testing=False).copy()  # Busca calendário oficial daquele ano (sem testes de pré-temporada)
    keep_cols = [c for c in ["RoundNumber","EventName","OfficialEventName","EventDate","Location","EventFormat"] if c in cal.columns]
                                                              # Define quais colunas manter (apenas as disponíveis)
    cal = cal[keep_cols]                                    # Seleciona apenas as colunas relevantes
    cal["Year"] = y                                         # Adiciona a coluna Year (ano) ao calendário
    cal["loc_lc"] = cal["Location"].fillna("").str.lower()  # Cria uma versão minúscula da Location para facilitar matching
    cal["ename_lc"] = cal["EventName"].fillna("").str.lower()  # Cria uma versão minúscula do EventName para matching
    all_events.append(cal)                                  # Acrescenta o calendário daquele ano à lista

events_df = pd.concat(all_events, ignore_index=True)        # Concatena todos os calendários em um único DataFrame

# =========================
# FUNÇÃO: Regra de correspondência linha-do-calendário ↔ alvo (circuito)
# =========================

def row_matches_target(row, target):                        # Define uma função para saber se a linha do calendário bate com o circuito
    loc = row["loc_lc"]                                     # Obtém a Location em minúsculas
    enm = row["ename_lc"]                                   # Obtém o EventName em minúsculas
    if any(sub in loc for sub in target["location"]):       # Se alguma das substrings de location aparecer na Location
        return True                                         # Então consideramos que bate
    if any(sub in enm for sub in target["event_sub"]):      # Ou se alguma substring de evento aparecer no EventName
        return True                                         # Também consideramos que bate
    return False                                            # Caso contrário, não bate

# =========================
# FUNÇÃO: Extrai LAPS da CORRIDA (session = 'R') de um evento
# =========================

def extract_event_race_laps(year: int, round_number: int | None, event_name: str | None):
    """
    Extrai o DataFrame de 'laps' da sessão de corrida ('R') para um evento.
    Usa round_number quando disponível; caso contrário, tenta com event_name.
    Retorna um DataFrame de voltas com metadados de rastreabilidade.
    """
    if pd.notna(round_number):                              # Se o número da etapa (RoundNumber) estiver disponível
        ses = fastf1.get_session(int(year), int(round_number), 'R')  # Busca a sessão 'R' pela combinação (ano, round)
    else:                                                   # Caso não haja RoundNumber
        ses = fastf1.get_session(int(year), str(event_name), 'R')    # Busca a sessão 'R' pelo nome do evento (menos robusto, mas funciona)

    ses.load()                                              # Carrega os dados da sessão (usa cache se existir; baixa caso não)
    laps = ses.laps.copy()                                  # Copia o DataFrame de voltas (FastF1 gera em ses.laps)

    laps["Year"] = year                                     # Anota o ano (para rastreabilidade)
    laps["RoundNumber"] = int(round_number) if pd.notna(round_number) else np.nan  # Anota o round (ou NaN)
    laps["EventName"] = ses.event.EventName if hasattr(ses, "event") else event_name  # Anota o nome oficial do evento
    laps["SessionName"] = "Race"                            # Marca explicitamente o nome da sessão (Race)

    return laps                                             # Retorna o DataFrame de voltas daquela corrida

# =========================
# LOOP PRINCIPAL: Para cada circuito, encontra eventos (por ano) e extrai voltas
# =========================

all_circuits_laps = []                                      # Lista para acumular as voltas de TODOS os circuitos (para dataset mestre)

for circuito, matchers in TARGETS.items():                  # Itera sobre cada circuito alvo e seus critérios de match
    mask = events_df.apply(lambda r: row_matches_target(r, matchers), axis=1)
                                                            # Aplica a função de matching a cada linha do calendário
    cal_hits = events_df[mask].copy().sort_values(["Year","RoundNumber","EventDate"])
                                                            # Filtra os eventos que batem e ordena por ano/etapa/data
    if cal_hits.empty:                                      # Se nenhum evento bateu para esse circuito
        print(f"[AVISO] Nenhum evento encontrado para '{circuito}' nos anos {YEARS}.")
                                                            # Loga um aviso (pode acontecer em anos sem corrida naquele circuito)
        continue                                            # Passa ao próximo circuito

    print(f"\n=== {circuito}: {len(cal_hits)} evento(s) no calendário 2022–2024 ===")  # Log informativo do circuito
    per_circuit_laps = []                                   # Lista para acumular as voltas de todos os anos daquele circuito
    errors = []                                             # Lista para registrar erros por ano/evento (se houver)

    for _, ev in cal_hits.iterrows():                       # Itera linha a linha pelos eventos que bateram com o circuito
        y = int(ev["Year"])                                 # Extrai o ano daquele evento
        rnd = ev["RoundNumber"] if "RoundNumber" in ev and pd.notna(ev["RoundNumber"]) else None
                                                            # Extrai RoundNumber (ou None se não existir)
        ename = ev["EventName"]                             # Extrai EventName (para fallback ou logging)
        try:                                                # Inicia bloco de tentativa de extração
            df_laps = extract_event_race_laps(y, rnd, ename)  # Extrai as voltas da corrida para aquele evento/ano
            df_laps["Circuito"] = circuito                  # Adiciona o rótulo do circuito (ex.: 'Monza', 'COTA', etc.)
            df_laps["EventDate"] = pd.to_datetime(ev["EventDate"]).date() if pd.notna(ev["EventDate"]) else pd.NaT
                                                            # Adiciona a data do evento (convertendo para date)
            df_laps["Location"] = ev["Location"]            # Adiciona a Location (cidade/pista)
            df_laps["OfficialEventName"] = ev.get("OfficialEventName", np.nan)
                                                            # Adiciona o nome oficial completo do evento (se existir)
            per_circuit_laps.append(df_laps)                # Acumula as voltas desse evento no buffer do circuito
            print(f"[OK] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {len(df_laps)} voltas.")
                                                            # Loga sucesso com contagem de voltas
        except Exception as e:                              # Em caso de erro na extração/carregamento
            print(f"[ERRO] {circuito} {y} (Round {rnd if rnd is not None else 'n/a'}): {e}")
                                                            # Loga o erro com detalhes
            errors.append((circuito, y, str(e)))            # Registra o erro para referência posterior

    if per_circuit_laps:                                    # Se coletamos alguma volta para esse circuito
        laps_circuito = pd.concat(per_circuit_laps, ignore_index=True)
                                                            # Concatena todas as voltas dos anos daquele circuito
        sort_cols = [c for c in ["Year","Driver","LapNumber"] if c in laps_circuito.columns]
                                                            # Define colunas de ordenação que existirem no DF
        if sort_cols:                                       # Se há colunas para ordenar
            laps_circuito = laps_circuito.sort_values(sort_cols).reset_index(drop=True)
                                                            # Ordena e reseta índice

        safe_circuit = circuito.lower().replace(" ", "_")   # Cria um nome seguro para arquivo (minúsculas e underscores)
        out_csv = INTERIM / f"{safe_circuit}_2022-2024_all_laps.csv"     # Caminho do CSV por circuito
        out_parq = INTERIM / f"{safe_circuit}_2022-2024_all_laps.parquet" # Caminho do Parquet por circuito

        laps_circuito.to_csv(out_csv, index=False, encoding="utf-8")     # Salva CSV (UTF-8)
        laps_circuito.to_parquet(out_parq, index=False)                  # Salva Parquet (requer pyarrow ou fastparquet)

        print(f"[SALVO] {circuito}:")                       # Loga que salvou os arquivos do circuito
        print(f"        CSV:     {out_csv}")                # Mostra caminho do CSV
        print(f"        Parquet: {out_parq}")               # Mostra caminho do Parquet

        all_circuits_laps.append(laps_circuito)             # Acumula no dataset mestre
    else:                                                   # Caso não tenha conseguido nenhuma volta para o circuito
        print(f"[AVISO] Nenhuma volta consolidada para '{circuito}'. Erros: {len(errors)}")
                                                            # Loga aviso com quantidade de erros

# =========================
# DATASET MESTRE (todos os circuitos juntos)
# =========================

if all_circuits_laps:                                       # Se pelo menos um circuito teve voltas extraídas
    master = pd.concat(all_circuits_laps, ignore_index=True)  # Concatena tudo em um único DataFrame mestre

    preview_cols = [c for c in ["Circuito","Year","EventName","Driver","LapNumber","LapTime","Compound","Stint","PitInTime","PitOutTime"] if c in master.columns]
                                                            # Define um conjunto de colunas para pré-visualização (apenas as que existirem)
    print("\n=== AMOSTRA (dataset mestre) ===")             # Título para a amostra
    if preview_cols:                                        # Se há colunas para exibir
        print(master[preview_cols].head(12).to_string(index=False))  # Imprime 12 linhas de amostra do mestre
    else:                                                   # Caso contrário
        print(master.head(12).to_string(index=False))       # Imprime 12 linhas com as colunas que existirem

    master_csv = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.csv"       # Caminho do CSV mestre
    master_parq = INTERIM / "ALLCIRCUITS_2022-2024_all_laps.parquet"  # Caminho do Parquet mestre
    master.to_csv(master_csv, index=False, encoding="utf-8")          # Salva CSV mestre
    master.to_parquet(master_parq, index=False)                       # Salva Parquet mestre

    print(f"\n[SALVO] Dataset mestre:")                    # Log de confirmação de salvamento do mestre
    print(f"        CSV:     {master_csv}")                # Mostra caminho do CSV mestre
    print(f"        Parquet: {master_parq}")               # Mostra caminho do Parquet mestre
else:                                                       # Se nenhum circuito retornou voltas
    raise RuntimeError("Nenhuma volta foi extraída para os circuitos solicitados.")
                                                            # Lança erro explícito para facilitar debug/atenção do usuário


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...



=== Bahrein: 3 evento(s) no calendário 2022–2024 ===


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

[OK] Bahrein 2022 (Round 1): 1125 voltas.


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

...

[ERRO] Abu Dhabi 2022 (Round 22): The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNIN

[ERRO] Abu Dhabi 2023 (Round 22): The data you are trying to access has not been loaded yet. See `Session.load`


logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
logger      WARNING 	Failed to load result data from Ergast!
core        WARNING 	No result data for this session available on Ergast! (This is expected for recent sessions)
core        WARNING 	Failed to load driver list and session results!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNIN

[ERRO] Abu Dhabi 2024 (Round 24): The data you are trying to access has not been loaded yet. See `Session.load`
[AVISO] Nenhuma volta consolidada para 'Abu Dhabi'. Erros: 3

=== AMOSTRA (dataset mestre) ===
Circuito  Year          EventName Driver  LapNumber                LapTime Compound  Stint PitInTime PitOutTime
 Bahrein  2022 Bahrain Grand Prix    ALB        1.0 0 days 00:01:47.636000     SOFT    1.0       NaT        NaT
 Bahrein  2022 Bahrain Grand Prix    ALB        2.0 0 days 00:01:40.548000     SOFT    1.0       NaT        NaT
 Bahrein  2022 Bahrain Grand Prix    ALB        3.0 0 days 00:01:40.664000     SOFT    1.0       NaT        NaT
 Bahrein  2022 Bahrain Grand Prix    ALB        4.0 0 days 00:01:41.126000     SOFT    1.0       NaT        NaT
 Bahrein  2022 Bahrain Grand Prix    ALB        5.0 0 days 00:01:42.303000     SOFT    1.0       NaT        NaT
 Bahrein  2022 Bahrain Grand Prix    ALB        6.0 0 days 00:01:41.708000     SOFT    1.0       NaT        NaT
 Bahrein 